# Experiment 01 — Resource-aware EfficientNet-B0 flood segmentation

This notebook runs the **first real experiment** for *Small Models, Honest Maps*.

**Model:** UNet + EfficientNet-B0  
**Data:** ETCI 2021 Sentinel-1 VV/VH + stabilized ratio channel  
**Train regions:** Nebraska, North Alabama, Bangladesh, Red River North  
**Held-out evaluation:** Florence  
**Training:** 60 epochs, physical batch 8, gradient accumulation 4 (effective batch 32), AMP, cosine LR schedule

The notebook intentionally starts with environment/data checks before launching the full run. The full result is only considered valid if the exact config and output artifacts are preserved.


## 0. Start a GPU runtime

In Colab: **Runtime → Change runtime type → GPU**.

A T4/L4-class free GPU is enough for this B0 experiment. The B7 reference model is **not** part of this run.


In [ ]:
!nvidia-smi
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory/1024**3:.1f} GB")


## 1. Clone the resource-aware branch and install dependencies

Until PR #3 is merged, this notebook explicitly checks out `resource-aware-experiments`.
After it is merged, changing `BRANCH` to `main` is sufficient.


In [ ]:
import os, shutil, subprocess, pathlib

REPO = "https://github.com/nazizahed/Uncertainty-Aware-Flood-Segmentation-from-Sentinel-1-for-Near-Real-Time-Applications.git"
BRANCH = "resource-aware-experiments"
WORKDIR = "/content/sar-flood-uq"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

!git clone --branch {BRANCH} --single-branch {REPO} {WORKDIR}
%cd {WORKDIR}
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
print("Repository ready.")


## 2. Mount Google Drive for experiment persistence

Only checkpoints/logs/results are mirrored to Drive. Training data stays on the local Colab disk for faster I/O.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/sar-flood-uq"
EXP_NAME = "lightweight_unet_efficientnetb0"
DRIVE_RUN = f"{DRIVE_ROOT}/runs/{EXP_NAME}"
os.makedirs(DRIVE_RUN, exist_ok=True)
print("Persistent run directory:", DRIVE_RUN)


## 3. Validate the repository before using GPU time

These checks are cheap and should pass before downloading/training.


In [ ]:
!python scripts/validate_repository.py
!python -m pytest -q


## 4. Download and inspect ETCI 2021

The dataset is downloaded to the Colab runtime for training speed. Review the official ETCI/NASA data terms and acknowledgement requirements before using the community mirror.


In [ ]:
DATA_ROOT = f"{WORKDIR}/data/etci"

if not os.path.exists(DATA_ROOT):
    !python scripts/download_data.py --out data/etci
else:
    print("Dataset already present:", DATA_ROOT)


In [ ]:
from pathlib import Path
import yaml, sys
sys.path.insert(0, "src")
from sarflood.data.dataset import ETCIFloodDataset

with open("configs/lightweight_unet_b0.yaml") as f:
    cfg = yaml.safe_load(f)

train_ds = ETCIFloodDataset(
    cfg["data"]["root"],
    cfg["data"]["regions"],
    cfg["data"]["bands"],
    rotation_aug=cfg["data"]["rotation_aug"],
    image_size=cfg["data"]["image_size"],
    ratio_clip=cfg["data"]["ratio_clip"],
)
val_ds = ETCIFloodDataset(
    cfg["data"]["root"],
    cfg["data"]["val_regions"],
    cfg["data"]["bands"],
    rotation_aug=False,
    image_size=cfg["data"]["image_size"],
    ratio_clip=cfg["data"]["ratio_clip"],
)

print(f"Base training tiles: {len(train_ds.records):,}")
print(f"Training samples after rotations: {len(train_ds):,}")
print(f"Florence validation tiles: {len(val_ds):,}")
print(f"Flood-positive exposed training samples: {train_ds.flood_flags.mean():.1%}")
sample = train_ds[0]
print("Input:", tuple(sample["image"].shape), "Mask:", tuple(sample["mask"].shape), "ID:", sample["id"])
assert sample["image"].shape[0] == 3
assert sample["mask"].shape[0] == 1


## 5. Confirm the exact experiment configuration

Do **not** silently change this config during the run. If memory problems require a smaller physical batch, keep the effective batch approximately fixed with additional accumulation and save the modified config as a distinct experiment.


In [ ]:
from pprint import pprint
pprint(cfg)

physical_batch = cfg["training"]["batch_size"]
accum = cfg["training"].get("gradient_accumulation_steps", 1)
print("Physical batch:", physical_batch)
print("Gradient accumulation:", accum)
print("Effective batch:", physical_batch * accum)
assert cfg["model"]["encoder"] == "efficientnet-b0"
assert physical_batch == 8
assert accum == 4


## 6. One-batch GPU sanity check

This checks that the real B0 model, real dataset, forward pass, loss, and backward pass fit in memory before starting the long run.


In [ ]:
import torch
from sarflood.data.dataset import build_dataloaders
from sarflood.models.build import build_model
from sarflood.training.losses import BCEDiceLoss

device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader, _ = build_dataloaders(cfg)
model = build_model(cfg["model"], in_channels=len(cfg["data"]["bands"])).to(device)
loss_fn = BCEDiceLoss()

batch = next(iter(train_loader))
img = batch["image"].to(device)
mask = batch["mask"].to(device)

model.train()
with torch.cuda.amp.autocast(enabled=(device == "cuda")):
    logits = model(img)
    loss = loss_fn(logits, mask) / cfg["training"].get("gradient_accumulation_steps", 1)
loss.backward()

print("Sanity loss:", float(loss.detach().cpu()))
if device == "cuda":
    print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Peak allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
del model, img, mask, logits, loss, batch, train_loader
if device == "cuda":
    torch.cuda.empty_cache()
print("One-batch sanity check passed.")


## 7. Full Experiment 01 training

The training script saves `best.pt`, `last.pt`, `log.csv`, and `config.yaml` locally. A lightweight background mirror copies those files to Drive every few minutes.

If Colab disconnects, the most recent mirrored artifacts remain in Drive.


In [ ]:
import os, shutil, threading

RUN_DIR = f"{WORKDIR}/runs/{EXP_NAME}"

def copy_run_artifacts():
    os.makedirs(DRIVE_RUN, exist_ok=True)
    for name in ["best.pt", "last.pt", "log.csv", "config.yaml"]:
        src = os.path.join(RUN_DIR, name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_RUN, name))

def mirror_loop(stop_event):
    copy_run_artifacts()
    while not stop_event.wait(180):
        copy_run_artifacts()

stop = threading.Event()
thread = threading.Thread(target=mirror_loop, args=(stop,), daemon=True)
thread.start()

try:
    !python scripts/train.py --config configs/lightweight_unet_b0.yaml
finally:
    stop.set()
    thread.join()
    copy_run_artifacts()

print("Persistent artifacts:", DRIVE_RUN)


## 8. Inspect the training history

We are looking for convergence and a sensible gap between training and held-out Florence performance—not just the best single epoch.


In [ ]:
import pandas as pd, os
log_path = os.path.join(RUN_DIR, "log.csv")
hist = pd.read_csv(log_path)
display(hist.tail(10))

best_idx = hist["eval_score"].idxmax()
print("Best epoch:")
display(hist.loc[[best_idx]])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(hist["epoch"], hist["train_loss"], label="train loss")
plt.plot(hist["epoch"], hist["val_loss"], label="Florence loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(hist["epoch"], hist["f1"], label="F1")
plt.plot(hist["epoch"], hist["iou"], label="pooled IoU")
plt.plot(hist["epoch"], hist["miou_tiles"], label="mean tile IoU")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True)
plt.legend()
plt.show()


## 9. Deterministic Florence evaluation

This is the first primary result. It produces segmentation metrics, calibration, and the one-pass deterministic uncertainty baselines.


In [ ]:
RESULT_DIR = f"{RUN_DIR}/results"
os.makedirs(RESULT_DIR, exist_ok=True)

!python scripts/evaluate.py \
  --checkpoint {RUN_DIR}/best.pt \
  --regions florence \
  --out {RESULT_DIR}/florence_deterministic.json

shutil.copy2(
    f"{RESULT_DIR}/florence_deterministic.json",
    os.path.join(DRIVE_RUN, "florence_deterministic.json")
)


In [ ]:
import json, pprint
with open(f"{RESULT_DIR}/florence_deterministic.json") as f:
    det = json.load(f)

print("Segmentation metrics")
pprint.pp(det["metrics"])
print("\nCalibration")
pprint.pp(det["calibration"])
print("\nSelective prediction: deterministic baselines")
pprint.pp({
    k: {"aurc": v["aurc"], "sparsification_error": v["sparsification_error"]}
    for k, v in det["selective_prediction"].items()
})


## 10. First stochastic-UQ checkpoint: MC dropout with 5 passes

We start at **5 passes**, not 20. We only escalate to 10 or 20 if the uncertainty ranking improves enough to justify the extra inference cost.


In [ ]:
!python scripts/evaluate.py \
  --checkpoint {RUN_DIR}/best.pt \
  --regions florence \
  --mc-passes 5 \
  --out {RESULT_DIR}/florence_mc5.json

shutil.copy2(
    f"{RESULT_DIR}/florence_mc5.json",
    os.path.join(DRIVE_RUN, "florence_mc5.json")
)


In [ ]:
with open(f"{RESULT_DIR}/florence_mc5.json") as f:
    mc5 = json.load(f)

summary = {
    name: {
        "aurc": values["aurc"],
        "sparsification_error": values["sparsification_error"],
    }
    for name, values in mc5["selective_prediction"].items()
}
pprint.pp(summary)


## 11. Decision gate

**Do not automatically run MC-10/MC-20 or the next model.**

At this point record:

- best Florence F1 / pooled IoU / mean-tile IoU;
- overall, flood, and non-flood calibration;
- deterministic entropy/confidence AURC;
- MC-5 predictive entropy / mutual-information / variance AURC;
- training time and peak GPU memory.

Then compare the reliability improvement of MC-5 against its ~5× inference cost. The result determines the next experiment.


In [ ]:
print("Experiment 01 complete.")
print("Artifacts saved in:", DRIVE_RUN)
print("Next decision: inspect B0 deterministic vs MC-5 results before spending more GPU time.")
